# This notebook creates the tables that will be uploaded to Databricks
### "Final Slurm Log" provides job metadata, and "Final Subset" provides the target job ID's.
### Databricks will use this information to pull in the correct CPU/GPU tables from AWS, and process all the new features.

In [1]:
import s3fs
fs = s3fs.S3FileSystem(anon=True)
import pandas as pd
pd.set_option('display.max_columns', None)
import numpy as np
import re

In [2]:
# Load the Slurm Log
df_slurm = pd.read_csv('s3://mit-supercloud-dataset/datacenter-challenge/202201/slurm-log.csv')
print(df_slurm.shape)

(395914, 29)


#
# Creating Final Slurm Log for Databricks

In [3]:
# Create new column for count of GPU's allocated for each job
def get_allocated_gpu_count(tres_str):
    tres_str = str(tres_str)
    mappings = dict(re.findall(r'(\d+)=(\d+)', tres_str))
    
    for specific_id in ['1001', '1002']:
        if specific_id in mappings:
            return int(mappings[specific_id])
    return 0

df_slurm['gpus_alloc'] = df_slurm['tres_alloc'].apply(get_allocated_gpu_count)

In [4]:
# Create new column for count of CPU's allocated for the job
def get_allocated_cpu_count(tres_str):
    tres_str = str(tres_str)
    mappings = dict(re.findall(r"(\d+)=(\d+)", tres_str))

    # TRES ID '1' represents CPUs
    if "1" in mappings:
        return int(mappings["1"])
    return 0

df_slurm["cpus_alloc"] = df_slurm["tres_alloc"].apply(get_allocated_cpu_count)

In [5]:
# Add binary column for whether job was successful
df_slurm['job_completed'] = df_slurm['state'] == 3

In [6]:
# Create new columns for job duration
df_slurm['duration_seconds'] = df_slurm['time_end'] - df_slurm['time_start']
df_slurm['duration_hours'] = df_slurm['duration_seconds'] / 3600
print(df_slurm[['id_job', 'time_start', 'time_end', 'duration_seconds', 'duration_hours']].head(3))

           id_job  time_start    time_end  duration_seconds  duration_hours
0  82691694838059  1609806297  1609806605               308        0.085556
1   3434806870797  1609806607  1609807004               397        0.110278
2   8370846758272  1609807004  1609807331               327        0.090833


In [7]:
# Saving final Slurm Log to CSV to be used in Databricks
df_slurm.to_csv('Final_Slurm_Log.csv', index=False)

#
# Creating Final Subset of Jobs for Analysis

In [8]:
# Filter out jobs that have more than 1 node allocated
df_slurm = df_slurm[df_slurm['nodes_alloc'] == 1]
print(df_slurm['nodes_alloc'].value_counts())

nodes_alloc
1    389353
Name: count, dtype: int64


In [9]:
# Create a subset that contains all jobs with duration over 2 hours which also failed
# Combine that subset to a random sample of jobs with duration over two hours which succeeded
# The final subset should have an equal number of jobs that failed and jobs that succeeded

df_long = df_slurm[df_slurm['duration_hours'] > 2].copy()
group_a = df_long[df_long['state'].isin([5, 6, 11])]
group_b_pool = df_long[df_long['state'] == 3]
sample_size = len(group_a)

if len(group_b_pool) >= sample_size:
    group_b_sample = group_b_pool.sample(n=sample_size, random_state=30)
else:
    group_b_sample = group_b_pool
    print(f"Warning: Only {len(group_b_pool)} successful jobs found.")

# 5. Combine both groups into the final subset
df_subset = pd.concat([group_a, group_b_sample])

print(f"Total jobs in subset: {len(df_subset)}")
print(df_subset['state'].value_counts())

Total jobs in subset: 30596
state
3     15298
6      6967
5      6084
11     2247
Name: count, dtype: int64


In [10]:
print(df_subset['job_completed'].value_counts())

job_completed
False    15298
True     15298
Name: count, dtype: int64


In [11]:
df_subset.tail(3)

,id_job,id_array_job,id_array_task,id_user,kill_requid,nodes_alloc,nodelist,cpus_req,derived_ec,exit_code,gres_used,array_max_tasks,array_task_pending,constraints,flags,mem_req,partition,priority,state,timelimit,time_submit,time_eligible,time_start,time_end,time_suspended,track_steps,tres_alloc,tres_req,job_type,gpus_alloc,cpus_alloc,job_completed,duration_seconds,duration_hours
234391,31362814217733,16618712154521,4294967294,30756960474747,61026541062099,1,['r6631426-n680758'],1,0,0,NaN,0,0,\N,4,9223372036854779808,xeon-p8,10095,3,4294967295,1623221328,1623221328,1623221331,1623292798,0,0,"1=1,2=4000,4=1,5=1","1=1,2=4000,4=1,5=1",OTHER,0,1,True,71467,19.851944
159323,10942763225706,56635494550104,1856,23247324987446,61026541062099,1,['r9175025-n386398'],1,0,0,NaN,0,0,xeon-g6,4,9223372036854784000,normal,10159,3,5760,1619231943,1619364158,1619364158,1619384472,0,0,"1=1,2=8192,4=1,5=1","1=1,2=8192,4=1,5=1",OTHER,0,1,True,20314,5.642778
182987,35932056838267,82645334129839,775,23247324987446,61026541062099,1,['r9720335-n386398'],1,0,0,NaN,0,0,xeon-g6,8,9223372036854784000,normal,10137,3,5760,1619747421,1619887303,1619887321,1619907475,0,0,"1=1,2=8192,4=1,5=1","1=1,2=8192,4=1,5=1",OTHER,0,1,True,20154,5.598333


In [12]:
# Create 4 separate dataframes, one for each quarter of the final subset
# Need to do this becuase Databricks Free tier does not provide enough resources to process all 30,000+ jobs at once

quarter_size = len(df_subset) // 4

q1 = df_subset.iloc[:quarter_size]
q2 = df_subset.iloc[quarter_size : 2 * quarter_size]
q3 = df_subset.iloc[2 * quarter_size : 3 * quarter_size]
q4 = df_subset.iloc[3 * quarter_size:]

print(f"Sizes: Q1={len(q1)}, Q2={len(q2)}, Q3={len(q3)}, Q4={len(q4)}")

Sizes: Q1=7649, Q2=7649, Q3=7649, Q4=7649


In [13]:
q1.head(1)

,id_job,id_array_job,id_array_task,id_user,kill_requid,nodes_alloc,nodelist,cpus_req,derived_ec,exit_code,gres_used,array_max_tasks,array_task_pending,constraints,flags,mem_req,partition,priority,state,timelimit,time_submit,time_eligible,time_start,time_end,time_suspended,track_steps,tres_alloc,tres_req,job_type,gpus_alloc,cpus_alloc,job_completed,duration_seconds,duration_hours
5,26940411750496,16618712154521,4294967294,78105663882022,61026541062099,1,['r9535192-n386398'],1,0,33280,NaN,0,0,xeon-g6,2,9223372036854784308,normal,110692,5,1440,1609808481,1609808481,1609808481,1609818623,0,0,"1=1,2=8500,4=1,5=1","1=1,2=8500,4=1,5=1",LLSUB:INTERACTIVE,0,1,False,10142,2.817222


In [14]:
q2.head(1)

,id_job,id_array_job,id_array_task,id_user,kill_requid,nodes_alloc,nodelist,cpus_req,derived_ec,exit_code,gres_used,array_max_tasks,array_task_pending,constraints,flags,mem_req,partition,priority,state,timelimit,time_submit,time_eligible,time_start,time_end,time_suspended,track_steps,tres_alloc,tres_req,job_type,gpus_alloc,cpus_alloc,job_completed,duration_seconds,duration_hours
222830,73162980168657,16618712154521,4294967294,26561648369279,61026541062099,1,['r4874959-n680758'],1,0,34560,NaN,0,0,\N,4,9223372036854779808,xeon-p8,10318,5,4294967295,1622183612,1622183612,1622183613,1622621187,0,0,"1=1,2=4000,4=1,5=1","1=1,2=4000,4=1,5=1",OTHER,0,1,False,437574,121.548333


In [15]:
q3.head(1)

,id_job,id_array_job,id_array_task,id_user,kill_requid,nodes_alloc,nodelist,cpus_req,derived_ec,exit_code,gres_used,array_max_tasks,array_task_pending,constraints,flags,mem_req,partition,priority,state,timelimit,time_submit,time_eligible,time_start,time_end,time_suspended,track_steps,tres_alloc,tres_req,job_type,gpus_alloc,cpus_alloc,job_completed,duration_seconds,duration_hours
5765,5789972683768,30716366989311,57,91741573397365,61026541062099,1,['r4858666-n851693'],6,0,0,NaN,0,0,xeon-g6,4,9223372036854784308,normal,10276,3,4294967295,1609971416,1609971416,1609971418,1609986749,0,0,"1=6,2=51000,4=1,5=6","1=6,2=51000,4=1,5=6",OTHER,0,6,True,15331,4.258611


In [17]:
q4.head(1)

,id_job,id_array_job,id_array_task,id_user,kill_requid,nodes_alloc,nodelist,cpus_req,derived_ec,exit_code,gres_used,array_max_tasks,array_task_pending,constraints,flags,mem_req,partition,priority,state,timelimit,time_submit,time_eligible,time_start,time_end,time_suspended,track_steps,tres_alloc,tres_req,job_type,gpus_alloc,cpus_alloc,job_completed,duration_seconds,duration_hours
222896,56256219083083,48607843887162,92,27491691635712,61026541062099,1,['r9535192-n976057'],1,0,0,NaN,0,0,xeon-g6,4,9223372036854779904,normal,110636,3,2880,1622052576,1622052577,1622185695,1622201190,0,0,"1=1,2=4096,4=1,5=1,1002=1","1=1,2=4096,4=1,5=1,1002=1",OTHER,1,1,True,15495,4.304167


In [18]:
# Save just the JobID column to CSV files for easy upload to Databricks
# We only need this column becuase it will be used to locate the CPU and GPU filenames for each job

q1[['id_job']].to_csv('Final_Joblist_Q1.csv', index=False)
q2[['id_job']].to_csv('Final_Joblist_Q2.csv', index=False)
q3[['id_job']].to_csv('Final_Joblist_Q3.csv', index=False)
q4[['id_job']].to_csv('Final_Joblist_Q4.csv', index=False)